# Building an ETL Pipeline

## What is ETL?
ETL—meaning **extract**, **transform**, **load** is a data integration process that combines, cleans and organizes data from multiple sources into a single, consistent data set for storage in a data warehouse, data lake or other target system.

## Why is ETL important?

Organizations today have both structured and unstructured data from various sources including:
- Customer data from online payment and customer relationship management (CRM) systems
- Inventory and operations data from vendor systems
- Sensor data from Internet of Things (IoT) devices
- Marketing data from social media and customer feedback
- Employee data from internal human resources systems

By applying the process of extract, transform, and load (ETL), individual raw datasets can be prepared in a format and structure that is more consumable for analytics purposes, resulting in more meaningful insights. For example, online retailers can analyze data from points of sale to forecast demand and manage inventory. Marketing teams can integrate CRM data with customer feedback on social media to study consumer behavior.

### Data warehouses

A data warehouse is a central repository that can store multiple databases. Within each database, you can organize your data into tables and columns that describe the data types in the table. The data warehouse software works across multiple types of storage hardware—such as solid state drives (SSDs), hard drives, and other cloud storage—to optimize your data processing.

## The ETL Pipeline
An ETL pipeline is a sequence of processes that moves data from one or more source systems to a destination system (often a data warehouse) through three major phases:
1. **Extract**: This step involves pulling data from the source systems
2. **Transform**: In this stage, the data is cleaned, enriched, and reshaped so it fits the analytical model. Common operations include:
    - Handling missing values
    - Changing data types (e.g., int → date)
    - Generating surrogate keys
    - Calculating fields (e.g., age from birthdate)
    - Removing duplicates
3. **Load**: This step loads the transformed data into the target system. The load process can be:
- Full Refresh: Drop and reload all data (simple but inefficient)
- Incremental Load: Load only new or changed data (preferred for large volumes)
- Merge/Upsert: Insert or update records based on keys and row differences.

![The ETL Process](images/etl_process.png)

## Snowflake Environment Preparation

Before we load any data, we need to create several Snowflake objects that support our ETL pipeline:

 -    ✅ Schemas for organizing the pipeline (RAW → STAGE → DW)

 -   📂 File stage for staging uploaded files

 -   🧾 File format for parsing .csv.gzip files

 -   🔢 Sequence objects for generating surrogate key

### Step-by-Step Snowflake Setup (Run in Snowflake)

1. Create a Database
```SQL
create database if not exists SAKILA_DW;
```
2. Create Required Schemas

```SQL
-- Use your database
USE DATABASE SAKILA_DW;
USE WAREHOUSE COMPUTE_WH;
```
3. Create the Schemas for stage of the pipeline

```SQL
-- Create schemas for each stage of the pipeline
CREATE SCHEMA IF NOT EXISTS RAW;
CREATE SCHEMA IF NOT EXISTS STAGE;
CREATE SCHEMA IF NOT EXISTS DW;
```
4. Create a File Format
```sql
CREATE OR REPLACE FILE FORMAT file_format_csv_gzip
  TYPE = 'CSV'
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  SKIP_HEADER = 1
  COMPRESSION = 'GZIP'
  NULL_IF = ('\\N', 'NULL', '');
```
5. Create a File Stage
```sql
-- Create a named file stage at database level
CREATE OR REPLACE STAGE file_stage
  FILE_FORMAT = file_format_csv_gzip
  COMMENT = 'Stage for loading gzipped CSVs from Sakila Extracts';
```

6. Create Surrogate Key Sequences

We will use sequences to auto-generate surrogate keys in dimension tables.
```sql
-- Customer surrogate key sequence
CREATE OR REPLACE SEQUENCE stage.customer_sk_seq START = 1 INCREMENT = 1;

-- Film surrogate key sequence
CREATE OR REPLACE SEQUENCE stage.film_sk_seq START = 1 INCREMENT = 1;
```